In [126]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

In [127]:
df = pd.read_csv('../data/IRIS.csv')

In [128]:
torch.manual_seed(42)

np.random.seed(42)

In [129]:
df.head(10)

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa
5,5.4,3.9,1.7,0.4,Iris-setosa
6,4.6,3.4,1.4,0.3,Iris-setosa
7,5.0,3.4,1.5,0.2,Iris-setosa
8,4.4,2.9,1.4,0.2,Iris-setosa
9,4.9,3.1,1.5,0.1,Iris-setosa


In [130]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


In [131]:
df.dropna(inplace=True)


In [132]:
df['species'].value_counts()

species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

In [133]:
X_raw = df.drop('species', axis=1).values

Y_raw = df['species'].values

In [134]:
df.head(10), df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


(   sepal_length  sepal_width  petal_length  petal_width      species
 0           5.1          3.5           1.4          0.2  Iris-setosa
 1           4.9          3.0           1.4          0.2  Iris-setosa
 2           4.7          3.2           1.3          0.2  Iris-setosa
 3           4.6          3.1           1.5          0.2  Iris-setosa
 4           5.0          3.6           1.4          0.2  Iris-setosa
 5           5.4          3.9           1.7          0.4  Iris-setosa
 6           4.6          3.4           1.4          0.3  Iris-setosa
 7           5.0          3.4           1.5          0.2  Iris-setosa
 8           4.4          2.9           1.4          0.2  Iris-setosa
 9           4.9          3.1           1.5          0.1  Iris-setosa,
 None)

In [135]:
X_raw

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [4.8, 3.4, 1.6, 0.2],
       [4.8, 3. , 1.4, 0.1],
       [4.3, 3. , 1.1, 0.1],
       [5.8, 4. , 1.2, 0.2],
       [5.7, 4.4, 1.5, 0.4],
       [5.4, 3.9, 1.3, 0.4],
       [5.1, 3.5, 1.4, 0.3],
       [5.7, 3.8, 1.7, 0.3],
       [5.1, 3.8, 1.5, 0.3],
       [5.4, 3.4, 1.7, 0.2],
       [5.1, 3.7, 1.5, 0.4],
       [4.6, 3.6, 1. , 0.2],
       [5.1, 3.3, 1.7, 0.5],
       [4.8, 3.4, 1.9, 0.2],
       [5. , 3. , 1.6, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [5.2, 3.5, 1.5, 0.2],
       [5.2, 3.4, 1.4, 0.2],
       [4.7, 3.2, 1.6, 0.2],
       [4.8, 3.1, 1.6, 0.2],
       [5.4, 3.4, 1.5, 0.4],
       [5.2, 4.1, 1.5, 0.1],
       [5.5, 4.2, 1.4, 0.2],
       [4.9, 3

In [136]:

from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_raw)


In [137]:
   
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
Y_encoder = label_encoder.fit_transform(Y_raw)

In [138]:
X_scaled, Y_encoder

(array([[-9.00681170e-01,  1.03205722e+00, -1.34127240e+00,
         -1.31297673e+00],
        [-1.14301691e+00, -1.24957601e-01, -1.34127240e+00,
         -1.31297673e+00],
        [-1.38535265e+00,  3.37848329e-01, -1.39813811e+00,
         -1.31297673e+00],
        [-1.50652052e+00,  1.06445364e-01, -1.28440670e+00,
         -1.31297673e+00],
        [-1.02184904e+00,  1.26346019e+00, -1.34127240e+00,
         -1.31297673e+00],
        [-5.37177559e-01,  1.95766909e+00, -1.17067529e+00,
         -1.05003079e+00],
        [-1.50652052e+00,  8.00654259e-01, -1.34127240e+00,
         -1.18150376e+00],
        [-1.02184904e+00,  8.00654259e-01, -1.28440670e+00,
         -1.31297673e+00],
        [-1.74885626e+00, -3.56360566e-01, -1.34127240e+00,
         -1.31297673e+00],
        [-1.14301691e+00,  1.06445364e-01, -1.28440670e+00,
         -1.44444970e+00],
        [-5.37177559e-01,  1.49486315e+00, -1.28440670e+00,
         -1.31297673e+00],
        [-1.26418478e+00,  8.00654259e-01, 

In [139]:
train_X, test_X, train_Y, test_Y = train_test_split(X_scaled, Y_encoder, test_size=0.2, random_state=42)

In [140]:
class IrisDataset(Dataset):
    def __init__(self, features, targets):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.Y = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

In [141]:
train_dataset = IrisDataset(train_X, train_Y)

test_dataset = IrisDataset(test_X, test_Y)

In [142]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

for batch_X, batch_y in train_loader:
    print("Batch X Shape:", batch_X.shape)
    print("Batch y Shape:", batch_y.shape)
    break

Batch X Shape: torch.Size([32, 4])
Batch y Shape: torch.Size([32])


In [143]:
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [144]:
class IrisModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(IrisModel, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 16)
        self.fc3 = nn.Linear(16, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [145]:
model = IrisModel(input_size=4, hidden_size=32, output_size=3)

In [146]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [147]:
epochs = 100

model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        optimizer.zero_grad()
        
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(train_loader):.4f}")

Epoch [1/100], Loss: 1.0544
Epoch [2/100], Loss: 1.0324
Epoch [3/100], Loss: 1.0165
Epoch [4/100], Loss: 0.9977
Epoch [5/100], Loss: 0.9782


Epoch [6/100], Loss: 0.9560
Epoch [7/100], Loss: 0.9328
Epoch [8/100], Loss: 0.8998
Epoch [9/100], Loss: 0.8723
Epoch [10/100], Loss: 0.8404
Epoch [11/100], Loss: 0.8188
Epoch [12/100], Loss: 0.7866
Epoch [13/100], Loss: 0.7580
Epoch [14/100], Loss: 0.7308
Epoch [15/100], Loss: 0.7072
Epoch [16/100], Loss: 0.6723
Epoch [17/100], Loss: 0.6439
Epoch [18/100], Loss: 0.6158
Epoch [19/100], Loss: 0.5911
Epoch [20/100], Loss: 0.5675
Epoch [21/100], Loss: 0.5401
Epoch [22/100], Loss: 0.5229
Epoch [23/100], Loss: 0.4978
Epoch [24/100], Loss: 0.4833
Epoch [25/100], Loss: 0.4624
Epoch [26/100], Loss: 0.4475
Epoch [27/100], Loss: 0.4280
Epoch [28/100], Loss: 0.4163
Epoch [29/100], Loss: 0.3997
Epoch [30/100], Loss: 0.3786
Epoch [31/100], Loss: 0.3694
Epoch [32/100], Loss: 0.3563
Epoch [33/100], Loss: 0.3425
Epoch [34/100], Loss: 0.3250
Epoch [35/100], Loss: 0.3231
Epoch [36/100], Loss: 0.3121
Epoch [37/100], Loss: 0.2964
Epoch [38/100], Loss: 0.2838
Epoch [39/100], Loss: 0.2792
Epoch [40/100], Lo

In [148]:
model.eval()

with torch.no_grad():
    correct = 0
    total = 0
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 100.00%


In [149]:
from sklearn.metrics import classification_report, confusion_matrix

cm = confusion_matrix(test_Y, predicted.numpy())
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]


In [152]:
torch.save(model.state_dict(), "../model/iris_model.pth")